# NASNetMobile — Neural Architecture Search Network (TensorFlow / Keras)
**Paper:** Learning Transferable Architectures for Scalable Image Recognition (Zoph et al., CVPR 2018)

| Property | Value |
|----------|-------|
| Framework | TensorFlow / Keras |
| Block type | NAS Normal Cell + Reduction Cell |
| Cell groups | 3 groups × 4 Normal Cells |
| Parameters | ~5.3M |
| Top-1 (ImageNet) | ~74.4% |
| Input size | 224×224 |
| Preprocessing | nasnet.preprocess_input → x/127.5 − 1.0 → [−1, 1] |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {bool(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# Cell 2 — NASNetMobile Architecture (from scratch)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ── Cell helpers ──────────────────────────────────────────────────────────────

def _sep_conv_bn(x, filters, kernel_size, strides=1):
    """Two stacked depthwise-separable convolutions with ReLU prefix and BN.
    Only the first SepConv uses the given stride; the second always uses stride=1.
    """
    x = layers.Activation('relu')(x)
    x = layers.SeparableConv2D(filters, kernel_size, strides=strides,
                                padding='same', use_bias=False)(x)
    x = layers.BatchNormalization(momentum=0.9997, epsilon=1e-3)(x)
    x = layers.Activation('relu')(x)
    x = layers.SeparableConv2D(filters, kernel_size, strides=1,
                                padding='same', use_bias=False)(x)
    x = layers.BatchNormalization(momentum=0.9997, epsilon=1e-3)(x)
    return x


def _adjust_block(p, ip, filters):
    """Adjust p so its spatial size and channel count match what the cell expects.
    Three cases:
      p is None          -> use ip as identity
      spatial mismatch   -> 2-branch strided avg-pool + conv to halve spatial dims
      channel mismatch   -> 1x1 conv projection to 'filters' channels
    """
    if p is None:
        return ip
    if ip.shape[1] != p.shape[1]:
        # p has 2x the spatial size of ip -> downsample with two offset branches
        p = layers.Activation('relu')(p)
        p1 = layers.AveragePooling2D(1, strides=2, padding='valid')(p)
        p1 = layers.Conv2D(filters // 2, 1, padding='same', use_bias=False)(p1)
        p2 = layers.ZeroPadding2D(padding=((0, 1), (0, 1)))(p)
        p2 = layers.Cropping2D(cropping=((1, 0), (1, 0)))(p2)
        p2 = layers.AveragePooling2D(1, strides=2, padding='valid')(p2)
        p2 = layers.Conv2D(filters // 2, 1, padding='same', use_bias=False)(p2)
        p = layers.Concatenate()([p1, p2])
        p = layers.BatchNormalization(momentum=0.9997, epsilon=1e-3)(p)
    elif p.shape[-1] != filters:
        p = layers.Activation('relu')(p)
        p = layers.Conv2D(filters, 1, padding='same', use_bias=False)(p)
        p = layers.BatchNormalization(momentum=0.9997, epsilon=1e-3)(p)
    return p


def _normal_cell(ip, p, filters):
    """NASNet-A Normal Cell (Figure 4, Zoph et al. 2018).
    Maintains spatial dimensions. Output has 6*filters channels.
    Returns: (cell_output, ip)  -- ip is passed through for the next cell's skip.

    5 parallel blocks added to adjusted-p:
      1: sep5x5(h)  + identity(h)
      2: sep5x5(p)  + sep3x3(h)
      3: avg3x3(h)  + p
      4: avg3x3(p)  + avg3x3(p)
      5: max3x3(h)  + sep3x3(p)
    Concat([p, x1, x2, x3, x4, x5])
    """
    p = _adjust_block(p, ip, filters)

    h = layers.Activation('relu')(ip)
    h = layers.Conv2D(filters, 1, padding='same', use_bias=False)(h)
    h = layers.BatchNormalization(momentum=0.9997, epsilon=1e-3)(h)

    x1 = layers.Add()([_sep_conv_bn(h, filters, 5), h])
    x2 = layers.Add()([_sep_conv_bn(p, filters, 5), _sep_conv_bn(h, filters, 3)])
    x3 = layers.Add()([layers.AveragePooling2D(3, strides=1, padding='same')(h), p])
    x4 = layers.Add()([
        layers.AveragePooling2D(3, strides=1, padding='same')(p),
        layers.AveragePooling2D(3, strides=1, padding='same')(p),
    ])
    x5 = layers.Add()([
        layers.MaxPooling2D(3, strides=1, padding='same')(h),
        _sep_conv_bn(p, filters, 3),
    ])
    return layers.Concatenate()([p, x1, x2, x3, x4, x5]), ip


def _reduction_cell(ip, p, filters):
    """NASNet-A Reduction Cell (Figure 4, Zoph et al. 2018).
    Halves spatial dimensions. Output has 4*filters channels.
    Returns: (cell_output, ip)

    5 blocks (first 3 use stride-2 ops on h/p):
      1: sep5x5_s2(h) + sep7x7_s2(p)
      2: max3x3_s2(h) + sep7x7_s2(p)
      3: avg3x3_s2(h) + sep5x5_s2(p)
      4: avg3x3_s1(x1) + x2
      5: sep3x3_s1(x1) + max3x3_s2(h)
    Concat([x2, x3, x4, x5])
    """
    p = _adjust_block(p, ip, filters)

    h = layers.Activation('relu')(ip)
    h = layers.Conv2D(filters, 1, padding='same', use_bias=False)(h)
    h = layers.BatchNormalization(momentum=0.9997, epsilon=1e-3)(h)

    x1 = layers.Add()([_sep_conv_bn(h, filters, 5, strides=2),
                        _sep_conv_bn(p, filters, 7, strides=2)])
    x2 = layers.Add()([layers.MaxPooling2D(3, strides=2, padding='same')(h),
                        _sep_conv_bn(p, filters, 7, strides=2)])
    x3 = layers.Add()([layers.AveragePooling2D(3, strides=2, padding='same')(h),
                        _sep_conv_bn(p, filters, 5, strides=2)])
    x4 = layers.Add()([layers.AveragePooling2D(3, strides=1, padding='same')(x1), x2])
    x5 = layers.Add()([_sep_conv_bn(x1, filters, 3),
                        layers.MaxPooling2D(3, strides=2, padding='same')(h)])
    return layers.Concatenate()([x2, x3, x4, x5]), ip


# ── Model ─────────────────────────────────────────────────────────────────────

def build_nasnetmobile(num_classes=1000, input_shape=(224, 224, 3)):
    """
    NASNetMobile — Neural Architecture Search Network (Mobile variant).
    Paper: Learning Transferable Architectures for Scalable Image Recognition
           Zoph et al., CVPR 2018.

    Architecture discovered by NAS on CIFAR-10 and transferred to ImageNet.
    NASNetMobile is the efficiency-optimised variant targeting mobile inference.

    Hyper-parameters:
      penultimate_filters = 1056
      filters             = 1056 // 24 = 44
      num_blocks          = 4  (Normal Cells per group)
      stem_filters        = 32
      filter_multiplier   = 2
      skip_reduction      = False

    Flow:
      Stem Conv3x3/2 (32 filters, padding=valid)
      Stem Reduction Cell  (filters*4 = 176)
      Stem Reduction Cell  (filters*2 = 88)
      Group 1: 4 Normal Cells (filters=44)       -> 6*44  = 264 ch
      Reduction Cell (filters*2=88,  skip_red=False)
      Group 2: 4 Normal Cells (filters*2=88)     -> 6*88  = 528 ch
      Reduction Cell (filters*4=176, skip_red=False)
      Group 3: 4 Normal Cells (filters*4=176)    -> 6*176 = 1056 ch
      ReLU -> GlobalAvgPool -> Dense(num_classes)
    """
    penultimate_filters = 1056
    num_blocks   = 4
    stem_filters = 32
    filt_mult    = 2
    filters      = penultimate_filters // 24   # 44

    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(stem_filters, 3, strides=2, padding='valid',
                      use_bias=False, name='stem_conv1')(inputs)
    x = layers.BatchNormalization(momentum=0.9997, epsilon=1e-3, name='stem_bn1')(x)

    p = None
    # Two stem reduction cells
    x, p = _reduction_cell(x, p, filters * filt_mult**2)   # filters*4 = 176
    x, p = _reduction_cell(x, p, filters * filt_mult)      # filters*2 = 88

    # Group 1: Normal cells (filters=44)
    for _ in range(num_blocks):
        x, p = _normal_cell(x, p, filters)

    # Reduction 1 (skip_reduction=False: p = returned ip = old x)
    x, p = _reduction_cell(x, p, filters * filt_mult)

    # Group 2: Normal cells (filters*2=88)
    for _ in range(num_blocks):
        x, p = _normal_cell(x, p, filters * filt_mult)

    # Reduction 2 (skip_reduction=False)
    x, p = _reduction_cell(x, p, filters * filt_mult**2)

    # Group 3: Normal cells (filters*4=176) -> 6*176=1056 channels
    for _ in range(num_blocks):
        x, p = _normal_cell(x, p, filters * filt_mult**2)

    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='nasnetmobile')


In [ ]:
NUM_CLASSES = 10
INPUT_SHAPE = (224, 224, 3)

model = build_nasnetmobile(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)
model.summary(line_length=120)

In [ ]:
dummy  = tf.random.normal((2, 224, 224, 3))
output = model(dummy, training=False)
print(f'Input  shape : {dummy.shape}')
print(f'Output shape : {output.shape}')

In [ ]:
total_params     = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
non_trainable    = total_params - trainable_params
print(f"{'='*48}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Non-trainable         : {non_trainable:,}")
print(f"{'='*48}")

In [ ]:
print(f"{'Layer':<55} {'Output Shape':<30} {'Params':>10}")
print('-' * 98)
total = 0
for layer in model.layers:
    params = layer.count_params()
    total += params
    try:
        out_shape = str(layer.output_shape)
    except Exception:
        out_shape = 'multiple'
    print(f"{layer.name:<55} {out_shape:<30} {params:>10,}")
print('-' * 98)
print(f"{'TOTAL':<86} {total:>10,}")

---
## Training

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    f'{DATA_DIR}/train', target_size=(224, 224),
    batch_size=BATCH_SIZE, class_mode='categorical',
)
val_gen = val_datagen.flow_from_directory(
    f'{DATA_DIR}/val', target_size=(224, 224),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False,
)
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'Classes    : {CLASS_NAMES}')
print(f'Train size : {train_gen.samples}')
print(f'Val size   : {val_gen.samples}')

In [ ]:
EPOCHS = 30

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'nasnetmobile_best.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.1, patience=5,
        min_lr=1e-7, verbose=1,
    ),
]

history = model.fit(
    train_gen, epochs=EPOCHS, validation_data=val_gen,
    callbacks=callbacks, verbose=1,
)
print('Best model saved: nasnetmobile_best.keras')

---
## Training Curves

In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NASNetMobile — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(epochs_range, history.history['loss'],         'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'],     'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', linewidth=2, markersize=4, label='Train Acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', linewidth=2, markersize=4, label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Inference

In [ ]:
def predict_image(image_path, top_k=5):
    img    = keras.utils.load_img(image_path, target_size=(224, 224))
    arr    = keras.utils.img_to_array(img) / 255.0
    tensor = tf.expand_dims(arr, 0)
    probs  = model(tensor, training=False)[0].numpy()
    top_idx = probs.argsort()[::-1][:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input')
    labels    = [CLASS_NAMES[i] for i in top_idx]
    top_probs = [probs[i] * 100 for i in top_idx]
    axes[1].barh(labels[::-1], top_probs[::-1])
    axes[1].set_xlabel('Confidence (%)'); axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout(); plt.show()
    print(f'Predicted: {CLASS_NAMES[top_idx[0]]}  ({probs[top_idx[0]]*100:.2f}%)')

# predict_image('your_image.jpg')
print('Call: predict_image("your_image.jpg")')

---
## ROC AUC Curve

In [ ]:
val_gen.reset()
y_pred     = model.predict(val_gen, steps=len(val_gen), verbose=1)
y_true     = val_gen.classes
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print(f'Predictions shape : {y_pred.shape}')
print(f'Labels shape      : {y_true_bin.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors  = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred[:, i])
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f'{CLASS_NAMES[i]}  (AUC = {auc(fpr,tpr):.3f})')

macro_auc = roc_auc_score(y_true_bin, y_pred, average='macro', multi_class='ovr')
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'NASNetMobile — ROC AUC  (Macro AUC = {macro_auc:.3f})', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Macro AUC : {macro_auc:.4f}')